# Explore the zoo results
Reads what `mmd analyse` and `mmd zoo` wrote to the store (`results/<cohort_version>/<model>/<pair>/`, figures under
`figures/` with the same layout). This notebook has no GPU or model code. Run it from the repo with the core
environment (`uv sync`, then pick the `.venv` kernel), or in Colab after `pip install -e .`.

Settings come from `mmd.toml` / `MMD_*`, the same as the CLI.

In [ ]:
from google.auth import default  # noqa: F401  (gcsfs uses your gcloud / Colab credentials)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mmd import config, fsio

cfg = config.load()
PAIR = 'cxr_notes'                       # or ecg_notes
models = sorted(p.rstrip('/').split('/')[-2] for p in fsio.glob(cfg._p('results', cfg.cohort_version, '*', PAIR))
                if '/zoo/' not in p)
print(cfg.root, cfg.cohort_version, PAIR, models)

## 1. The zoo table
One row per model × tier × readout × classifier:
- `info_share_note`: the note's share of the outcome information, from the two single-modality arms;
- `reliance_share_note`: the note's share of the counterfactual |Δ risk|;
- `bias` = reliance − information. Above 0 means the model leans on the note more than the note's information justifies.

`subset = official split` repeats every model on stays whose baseline film is outside the MIMIC-CXR train split,
which matters for MIMIC-exposed models.

In [ ]:
zoo = fsio.read_parquet(cfg._p('results', cfg.cohort_version, 'zoo', f'zoo_{PAIR}.parquet'))
cols = ['model', 'fusion', 'domain', 'mimic_exposure', 'subset', 'level_id', 'readout', 'clf', 'auroc_signal_only',
        'auroc_note_only', 'auroc_both', 'info_share_note', 'reliance_share_note', 'reliance_lo', 'reliance_hi', 'bias']
zoo[[c for c in cols if c in zoo]].round(3)

## 2. Reliance against information
Each point is one model, using the raw P(Yes) readout (behaviour) on the chosen tier. Points above the diagonal lean on
the note more than its information share justifies. Filled markers have no known MIMIC exposure; hollow markers were
trained on MIMIC data.

In [ ]:
LEVEL, CLF, SUBSET = 'L1', 'no probe (raw P(Yes))', 'all'
t = zoo[(zoo.level_id == LEVEL) & (zoo.clf == CLF) & (zoo.subset == SUBSET)].dropna(subset=['reliance_share_note'])
fig, ax = plt.subplots(figsize=(5.5, 5.2))
ax.plot([0, 1], [0, 1], color='#bdbdbd', ls='--', lw=1)
for r in t.itertuples():
    exposed = r.mimic_exposure not in (None, 'none')
    ax.errorbar(r.info_share_note, r.reliance_share_note,
                yerr=[[r.reliance_share_note - r.reliance_lo], [r.reliance_hi - r.reliance_share_note]],
                fmt='o', ms=7, capsize=3, mfc='white' if exposed else '#2a78d6', color='#2a78d6')
    ax.annotate(r.model, (r.info_share_note, r.reliance_share_note), textcoords='offset points', xytext=(6, 4),
                fontsize=8)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('note share of the outcome information (single-modality arms)')
ax.set_ylabel('note share of the counterfactual |Δ risk|')
ax.set_title(f'{LEVEL} · {CLF} · {SUBSET}', loc='left', fontsize=10)
plt.show()

## 3. Per-model tables
The same tables the notebooks write (`swap4_*`), for any model in the store.

In [ ]:
MODEL = models[0]
d = cfg.results_dir(MODEL, PAIR)
summary = fsio.read_parquet(f'{d}/swap4_summary_{PAIR}_clean.parquet')
auroc = fsio.read_parquet(f'{d}/swap4_auroc_{PAIR}.parquet')
paired = fsio.read_parquet(f'{d}/swap4_paired_dominance_{PAIR}.parquet')
by_draw = fsio.read_parquet(f'{d}/swap4_auroc_by_draw_{PAIR}.parquet')
summary[['level_id', 'readout', 'clf', 'family', 'slot', 'n', 'delta_died', 'delta_survived', 'gap', 'gap_lo',
         'gap_hi']].round(4)

In [ ]:
# single donor vs mean of donors (the matched-swap AUROC artefact): report the single-donor numbers
kind = np.where(by_draw.scored == 'mean of draws', 'mean of draws', 'one donor (avg over draws)')
by_draw.assign(kind=kind).pivot_table(index=['readout', 'level_id', 'family', 'slot'], columns='kind',
                                      values='d_auroc').round(3)

## 4. Figures
The charts `mmd analyse` saved for a model.

In [ ]:
from IPython.display import Image, display
figs = fsio.glob(f'{cfg.fig_dir(MODEL, PAIR)}/*.png')
print(len(figs), 'figures')
for p in figs[:4]:
    with fsio.open_(p) as f:
        display(Image(f.read()))